# Day 5 — Exercises: Visualisation
## Master's Course: Quantitative Skills for Biology

**Topics covered:** Matplotlib `fig, ax` pattern, line/bar/histogram/scatter plots, seaborn boxplots and heatmaps, labels and titles, subplots, saving figures, volcano plot.

> Every plot must have: a **title**, labelled **axes**, and a **legend** where applicable.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid", font_scale=1.1)
np.random.seed(42)

# ── Recreate the expression dataset from Day 4 ──────────────────────────────
n_genes      = 20
gene_names   = [f"GENE_{i:03d}" for i in range(1, n_genes + 1)]
sample_names = [f"ctrl_{i}" for i in range(1, 4)] + [f"treat_{i}" for i in range(1, 4)]
ctrl_cols    = ["ctrl_1", "ctrl_2", "ctrl_3"]
treat_cols   = ["treat_1", "treat_2", "treat_3"]

ctrl_data    = np.random.normal(8.0, 1.0, (n_genes, 3))
treat_mean   = np.full(n_genes, 8.0); treat_mean[:5] = 10.0
treat_data   = np.random.normal(treat_mean[:, None], 1.0, (n_genes, 3))

df = pd.DataFrame(np.hstack([ctrl_data, treat_data]),
                  index=gene_names, columns=sample_names)
df.index.name = "gene"
df["mean_ctrl"]  = df[ctrl_cols].mean(axis=1)
df["mean_treat"] = df[treat_cols].mean(axis=1)
df["log2fc"]     = np.log2((df["mean_treat"] + 0.01) / (df["mean_ctrl"] + 0.01))
df["variance"]   = df[sample_names].var(axis=1)

np.random.seed(7)
df["pvalue"] = np.where(
    df.index.isin([f"GENE_{i:03d}" for i in range(1, 6)]),
    np.random.uniform(0.0001, 0.04, n_genes)[:5],
    np.random.uniform(0.05, 1.0,    n_genes)[5:]
)

print("Dataset ready. Shape:", df.shape)

---
## Exercise 1 — Histogram: Effect of Log2 Transformation

Create a **side-by-side figure** (`fig, axes = plt.subplots(1, 2)`) showing:
- **Left panel:** histogram of the raw (linear) expression values (`2 ** df[sample_names].values.flatten()`)
- **Right panel:** histogram of the log2 expression values (`df[sample_names].values.flatten()`)

Requirements:
- Both panels: title, x-label, y-label
- Add a vertical dashed line at the mean of each distribution
- Use `plt.suptitle()` for an overall figure title

In [ ]:
log2_vals = df[sample_names].values.flatten()
raw_vals  = 2 ** log2_vals

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left panel — raw values

# Right panel — log2 values

plt.suptitle("Effect of log2 transformation", fontsize=13)
plt.tight_layout()
plt.show()

---
## Exercise 2 — Bar Chart: Top and Bottom Genes

Create a **horizontal bar chart** showing the **5 most upregulated** and **5 most downregulated** genes by `log2fc`.

Requirements:
- Colour upregulated bars `"tomato"`, downregulated bars `"steelblue"`
- Add a vertical dashed line at `x = 0`
- Label each bar with its log2fc value (use `ax.text()`)
- Title, x-label, y-label

In [ ]:
top5_up   = df.nlargest(5,  "log2fc")
top5_down = df.nsmallest(5, "log2fc")
combined  = pd.concat([top5_down, top5_up])   # down first so up appears at top

fig, ax = plt.subplots(figsize=(7, 5))

# Your code here

plt.tight_layout()
plt.show()

---
## Exercise 3 — Scatter Plot: Replicate Correlation

Create a scatter plot comparing **two treatment replicates** (`treat_1` vs `treat_2`):

1. Compute and display the Pearson correlation coefficient `r` in the title
2. Colour the **top 5 most variable genes** in orange; the rest in grey
3. Add a dashed diagonal line (y = x) representing perfect correlation
4. Label the top 3 variable genes by name using `ax.annotate()`

In [ ]:
x = df["treat_1"]
y = df["treat_2"]

# Pearson r
r, p = stats.pearsonr(x, y)

top5_var = df.nlargest(5, "variance").index

fig, ax = plt.subplots(figsize=(5, 5))

# Your scatter plot here

plt.tight_layout()
plt.show()

---
## Exercise 4 — Seaborn Boxplot: Per-Sample Distribution

Convert the expression DataFrame to **long format** and create a seaborn boxplot showing the distribution of expression values for each sample, coloured by condition (control vs treated).

Then add a **strip plot** on top (use `sns.stripplot()` with `dodge=False`, `alpha=0.5`, `jitter=True`) to show the individual data points.

*Hint: Melt with `df[sample_names].reset_index().melt(id_vars='gene', ...)`*

In [ ]:
# Melt to long format
df_long = df[sample_names].reset_index().melt(
    id_vars="gene", var_name="sample", value_name="expression"
)
df_long["condition"] = df_long["sample"].apply(
    lambda s: "Control" if s.startswith("ctrl") else "Treated"
)

fig, ax = plt.subplots(figsize=(8, 5))

# Boxplot

# Strip plot on top

ax.set_title("Expression distribution per sample")
plt.tight_layout()
plt.show()

---
## Exercise 5 — Volcano Plot

A **volcano plot** visualises differential expression results:
- x-axis: `log2fc`
- y-axis: `−log10(pvalue)` (large values = more significant)

Build the plot with these requirements:
1. Colour points: **red** if |log2fc| > 1 AND pvalue < 0.05 (significant), **grey** otherwise
2. Draw dashed threshold lines at x = ±1 and y = −log10(0.05)
3. Label the top 3 most significant genes by name
4. Add a legend, title, and axis labels

In [ ]:
log2fc    = df["log2fc"]
neg_log10 = -np.log10(df["pvalue"])

# Significance mask
is_sig = (df["pvalue"] < 0.05) & (df["log2fc"].abs() > 1)

fig, ax = plt.subplots(figsize=(7, 5))

# Non-significant points

# Significant points

# Threshold lines

# Label top 3 significant genes
top3_sig = df[is_sig].nsmallest(3, "pvalue")
for gene in top3_sig.index:
    pass   # ax.annotate(...)

ax.set_xlabel("log2 Fold-Change")
ax.set_ylabel("-log10(p-value)")
ax.set_title("Volcano Plot")
plt.tight_layout()
plt.show()

---
## Exercise 6 — Mini Project: Expression Story (4-Panel Figure)

Produce a **2×2 multi-panel figure** that tells the full story of the dataset. Each panel must be fully labelled.

| Panel | Content |
|-------|---------|
| Top-left | Histogram of all log2 expression values |
| Top-right | Volcano plot (from Exercise 5) |
| Bottom-left | Heatmap (z-score) of the top 10 most variable genes |
| Bottom-right | Bar chart of mean expression for upregulated vs non-DE genes |

Save the final figure as `expression_story.png` at 150 dpi.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 11))

# ── Top-left: histogram ──────────────────────────────────────────────────────


# ── Top-right: volcano plot ──────────────────────────────────────────────────


# ── Bottom-left: heatmap ─────────────────────────────────────────────────────
# Hint: use sns.heatmap() with cmap="RdBu_r" on z-scored data
top10 = df.nlargest(10, "variance").index
heat_z = df.loc[top10, sample_names].apply(stats.zscore, axis=1)


# ── Bottom-right: bar chart ───────────────────────────────────────────────────
is_up = (df["log2fc"] > 1) & (df["pvalue"] < 0.05)


plt.suptitle("Week 1 Expression Data — Summary", fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig("expression_story.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: expression_story.png")